# Tiny ReAct agent: notebook starter

This notebook uses LangChain's `create_agent` (a LangGraph agent loop). Edit the configuration cells, then run from top to bottom. The default provider is your local Ollama installation. The example uses `llama3.1:latest` for tool gathering and `qwen3-coder:30b` for structured assessments; both can be changed in `MODEL_CONFIG`.

The agent first gathers observed skills, guidelines, and data-tool results. Code assigns a short ID and deterministic locator to every data-source evidence span, retaining neighboring text or JSON parent context. The assessment model examines each data source separately against every observed labeled criterion, then reconciles all sources in a case-level pass. Each assessment cites one or more evidence IDs when evidence is available. Statuses distinguish `matched`, `no_match`, `insufficient_evidence`, and `not_applicable`. The final result includes source and criterion coverage so omissions remain visible.

From the project folder, install the notebook dependencies with `pip install -e ".[notebook]"` in the interpreter used by this kernel. Ollama must be running.


In [ ]:
from pathlib import Path
import json
from urllib.request import urlopen

from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import AIMessage, ToolMessage
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from pydantic import RootModel

from tiny_skill_agent import AssessmentConfig, assess
from tiny_skill_agent.adapters import inputs_from_messages, make_langchain_assessor
from tiny_skill_agent.assessment import CoverageDraft, PredicateDraft, Finding
from tiny_skill_agent.evidence import EvidenceMetadataMiddleware, tag_tool
from tiny_skill_agent.agent import DEFAULT_SKILLS_DIR, make_tools, skill_catalog

PROJECT_DIR = next((folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "pyproject.toml").is_file() and (folder / "src" / "tiny_skill_agent").is_dir()), None)
assert PROJECT_DIR is not None, "Run this notebook from the project folder or its notebooks folder."
print(f"Project: {PROJECT_DIR}")


## 1. Model configuration

Use `provider="ollama"` for local models. Set `model` for the ReAct tool loop and `assessment_model` for structured decisions; set the latter to `None` to reuse the first model. To use OpenAI, change `provider` to `"openai"`, select available model names, and set `OPENAI_API_KEY` in the notebook kernel environment.


In [ ]:
MODEL_CONFIG = {
    "provider": "ollama",                 # "ollama" or "openai"
    "model": "llama3.1:latest",           # ReAct tool-calling model
    "assessment_model": "qwen3-coder:30b", # More reliable structured assessment; set None to reuse model
    "base_url": "http://127.0.0.1:11434",
    "temperature": 0,
    "max_output_tokens": 4096,
}


## 2. Check the local models

This cell lists available Ollama models and checks both configured names. It is skipped when `provider="openai"`.


In [ ]:
if MODEL_CONFIG["provider"] == "ollama":
    with urlopen(f'{MODEL_CONFIG["base_url"].rstrip("/")}/api/tags', timeout=15) as response:
        installed = [item["name"] for item in json.load(response)["models"]]
    print("Installed Ollama models:", ", ".join(installed))
    for selected in {MODEL_CONFIG["model"], MODEL_CONFIG["assessment_model"] or MODEL_CONFIG["model"]}:
        assert selected in installed, f'{selected!r} is not installed. Run: ollama pull {selected}'


## 3. Skills

The agent discovers one `SKILL.md` per subfolder of the configured `SKILLS_DIR`. The starter includes `underwriting-demo`. Put each classified criterion on its own line, beginning with `[strength]` or `[concern]`. Criteria can also come from the system prompt or a directive tool. The output validator copies classification from the cited line, regardless of source kind. Add a skill directly on disk, or fill in `SKILL_DEFINITIONS` below. Entries in this cell are written to `SKILLS_DIR/<name>/SKILL.md` each time you run it.

In [ ]:
SKILLS_DIR = DEFAULT_SKILLS_DIR  # Bundled examples; change to a writable custom directory if desired
SKILL_DEFINITIONS = {
    # "my-skill": {
    #     "description": "When this skill should be used.",
    #     "instructions": "Step 1: ...\nStep 2: ...",
    # },
}

for name, spec in SKILL_DEFINITIONS.items():
    if not re.fullmatch(r"[a-z0-9][a-z0-9-]*", name):
        raise ValueError(f"Invalid skill name: {name!r}")
    folder = SKILLS_DIR / name
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "SKILL.md").write_text(
        f'---\nname: {name}\ndescription: {spec["description"]}\n---\n\n'
        + spec["instructions"].strip() + "\n",
        encoding="utf-8",
    )

print("Available skills:", ", ".join(sorted(skill_catalog(SKILLS_DIR))) or "(none)")

## 4. Tools and `_meta.kind`

Tag an ordinary LangChain tool with `tag_tool(tool, "directive")` or `tag_tool(tool, "data")`. The source kind is `tool` for ordinary tools and `skill` for `read_skill`; the system prompt is recorded as `system_prompt`. `EvidenceMetadataMiddleware` attaches registered metadata to successful tool messages. Each observed data-tool result becomes a distinct source for the atomic assessment pass. Add new tools to `CUSTOM_TOOLS`.


In [ ]:
@tool
def get_underwriting_facts() -> str:
    """Return illustrative facts from a sample application."""
    facts = [
        "Cash reserves: 8 months",
        "Missed payments in past 12 months: 2",
        "Debt-to-income ratio: 28%",
        "Income verification: unverified",
    ]
    return json.dumps(facts)


@tool
def get_guidelines() -> str:
    """Return an illustrative underwriting criterion."""
    return "[strength] Classify a debt-to-income ratio of 35% or less as a strength."


tag_tool(get_underwriting_facts, "data")
tag_tool(get_guidelines, "directive")
CUSTOM_TOOLS = [get_underwriting_facts, get_guidelines]
TOOLS = make_tools(SKILLS_DIR) + CUSTOM_TOOLS
print("Tools:", ", ".join(f'{tool.name} ({tool.metadata["_meta"]["kind"]})' for tool in TOOLS))

## 5. System prompt and agent settings

Put each system directive on its own line if you want it recorded in the evidence ledger. A system criterion must begin with `[strength]` or `[concern]` to support a classified finding. Its `criterion_source` is `system_prompt`. `recursion_limit` caps the graph steps in one request.

In [ ]:
SYSTEM_PROMPT = "\n".join([
    "You are a helpful assistant.",
    "Use tools to verify facts before reporting them.",
    "For tasks that may match a skill, call list_skills or read_skill before applying it.",
    "When the user names a skill, call read_skill for it before any other tools.",
    "Do not call the same tool with the same arguments more than once per request.",
    "Follow relevant skill instructions within the user's request.",
    "Do not claim to have used a skill or tool unless you did.",
    "[concern] Classify unverified income as a concern.",
])

AGENT_CONFIG = {"recursion_limit": 12, "show_tool_steps": True}

ASSESSMENT_CONFIG = AssessmentConfig(
    batch_size=1,
    retry_batch_size=1,
    max_retries=2,
    confirm_no_match=True,
    review_decisions=True,
)
SOURCE_CONFIG = {
    "labels": ("strength", "concern"),
    "allowed_source_kinds": ("skill", "tool", "system_prompt"),
    "allowed_source_ids": None,
}


## 6. Output schema

`PredicateDraft`, `CoverageDraft`, and `Finding` are reusable Pydantic models. The model quotes the criterion trigger and evaluates scope, condition truth and exceptions; code derives status and basis. Matched assessments contain separate findings with subjects, citations and reasons. Every criterion receives an independent review of the original evidence. Final negative decisions also retain a focused evidence-sufficiency check. Missing reviews, unverified negatives, or disagreements on status/finding citation groups block readiness. Inspect `pre_review_assessments`, `decision_reviews`, `no_match_reviews`, and `coverage.review_disagreements`. Findings are provisional whenever `decision_ready` is false. Code expands citations and classification; semantic judgments and calculations remain model-generated. The core accepts arbitrary classification labels. Set `USE_OUTPUT_SCHEMA = False` for plain text.


In [ ]:
# Inspect PredicateDraft.model_json_schema(), CoverageDraft.model_json_schema(), or Finding.model_json_schema()
# to see the model-facing and enriched output contracts.
EnrichedOutput = RootModel[list[Finding]]
USE_OUTPUT_SCHEMA = True


## 7. Build the agent

This creates the LangGraph-backed ReAct loop and a structured-output model. The structured model is called for each source/criterion pair, with bounded retries for missing assessments, then for case-level reconciliation. Rerun this cell after changing the model, tools, skills, system prompt, or schema.


In [ ]:
provider = MODEL_CONFIG["provider"].lower()
assessment_name = MODEL_CONFIG["assessment_model"] or MODEL_CONFIG["model"]
if provider == "ollama":
    def ollama_model(name):
        return ChatOllama(
            model=name, base_url=MODEL_CONFIG["base_url"],
            temperature=MODEL_CONFIG["temperature"],
            num_predict=MODEL_CONFIG["max_output_tokens"],
        )
    model = ollama_model(MODEL_CONFIG["model"])
    assessment_model = model if assessment_name == MODEL_CONFIG["model"] else ollama_model(assessment_name)
elif provider == "openai":
    def openai_model(name):
        return ChatOpenAI(
            model=name, temperature=MODEL_CONFIG["temperature"],
            max_tokens=MODEL_CONFIG["max_output_tokens"],
        )
    model = openai_model(MODEL_CONFIG["model"])
    assessment_model = model if assessment_name == MODEL_CONFIG["model"] else openai_model(assessment_name)
else:
    raise ValueError(f"Unsupported provider: {provider}")

agent = create_agent(
    model=model, tools=TOOLS, system_prompt=SYSTEM_PROMPT,
    middleware=[EvidenceMetadataMiddleware(TOOLS)],
)
assessment_callable = (
    make_langchain_assessor(assessment_model, method="json_schema" if provider == "ollama" else None)
    if USE_OUTPUT_SCHEMA else None
)
print(f"Agent ready: {provider} / {MODEL_CONFIG['model']} (assessment: {assessment_name})")


## 8. Input prompt and run

The notebook gathers tool results, converts them through `inputs_from_messages`, and calls `assess`. `source_coverage` checks each source/criterion pair; `coverage` checks final case assessments and reviews. Readiness requires complete coverage, sufficient evidence, nonempty criteria, and agreement with an independent review on status and finding citation groups. A reviewer correction remains non-ready until manually resolved. These flags do not prove semantic correctness. Custom LangGraph integrations can use `make_assessment_node`; see `docs/INTEGRATION.md`.


In [ ]:
def run_with_provenance(input_prompt, previous_messages=None):
    result = agent.invoke(
        {"messages": [*(previous_messages or []), {"role": "user", "content": input_prompt}]},
        config={"recursion_limit": AGENT_CONFIG["recursion_limit"]},
    )
    if AGENT_CONFIG["show_tool_steps"]:
        for message in result["messages"]:
            if isinstance(message, AIMessage) and message.tool_calls:
                for call in message.tool_calls:
                    print(f'CALL {call["name"]}: {call["args"]}')
            elif isinstance(message, ToolMessage):
                print(f"RESULT {message.name}: {str(message.content)[:300]}")
    if assessment_callable is None:
        display(Markdown(str(result["messages"][-1].content)))
        return result

    inputs = inputs_from_messages(result["messages"], TOOLS, system_prompt=SYSTEM_PROMPT, **SOURCE_CONFIG)
    assessment = assess(
        criteria=inputs.criteria, evidence=inputs.evidence,
        model=assessment_callable, config=ASSESSMENT_CONFIG, request=input_prompt,
    )
    result.update(assessment.model_dump(mode="json"))
    result["structured_response"] = EnrichedOutput(assessment.findings)
    print("Source coverage:", json.dumps(assessment.source_coverage, indent=2))
    print("Criterion coverage:", json.dumps(assessment.coverage, indent=2))
    print("Disagreements:", json.dumps(assessment.disagreements, indent=2))
    print(result["structured_response"].model_dump_json(indent=2))
    return result


INPUT_PROMPT = (
    "First read the underwriting-demo skill. Then call get_guidelines and "
    "get_underwriting_facts once each. Assess all four sample facts against "
    "the applicable criteria."
)
result = run_with_provenance(INPUT_PROMPT)
conversation = result["messages"]


## Optional: continue the conversation

Uncomment and edit the cell below. It sends the previous messages plus a new user prompt; history lasts only while this notebook kernel is running.

In [ ]:
# FOLLOW_UP = "Reassess the sample applicant with updated facts."
# result = run_with_provenance(FOLLOW_UP, conversation)
# conversation = result["messages"]